In [1]:
import logging
from einops import rearrange,einsum

import torch
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
# Create a console handler
console_handler = logging.StreamHandler()
# Create a formatting layout
formatter = logging.Formatter('%(name)s - %(levelname)s - %(asctime)s - %(message)s')
console_handler.setFormatter(formatter)

# Add the handler to your logger
logger.addHandler(console_handler)

In [105]:

import math
import torch

test = torch.zeros(5,4)
din = 20
dout = 29
sigma2 = 2/(din+dout)
sigma = math.sqrt(sigma2)
test2 = torch.nn.init.trunc_normal_(test,-3*sigma,3*sigma)
print(f"Test 2 is {test2}")
# torch.randn()
class Linear(torch.nn.Module):
    def __init__(self,in_features:int,out_features:int,device=None,dtype=None):
        super().__init__()
        sigma = math.sqrt(2/(in_features+out_features))
        # weight =
        self.weight = torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.zeros(in_features,out_features),-3*sigma,3*sigma))
        print(f"weights are {self.weight}")

    def forward(self,x:torch.Tensor)->torch.Tensor:
        return einsum(x,self.weight,"... d_in,d_in d_out -> ... d_out")

linear = Linear(5,4)
torch.rand(5)
linear(torch.rand(5))

Test 2 is tensor([[-6.4018e-01, -1.4084e+00, -1.6705e-01, -1.6542e-01],
        [-2.9487e-01, -4.7481e-01, -9.6387e-04, -8.9656e-01],
        [ 1.5192e-01, -1.2480e+00, -2.9752e-01, -1.0210e+00],
        [-4.7020e-01, -2.3237e-01, -1.5700e+00, -6.5179e-02],
        [-1.4416e+00, -1.8461e-02,  3.1841e-01, -5.2797e-01]])
weights are Parameter containing:
tensor([[-1.3855, -1.2296,  0.5404,  0.2833],
        [ 1.6663, -1.3255, -1.5077,  0.4343],
        [-1.4973,  0.4525,  1.3157, -1.9892],
        [-1.9393, -0.6576,  0.5057, -0.5143],
        [-0.7835, -1.4120, -0.5954, -1.3077]], requires_grad=True)


tensor([-4.2460, -2.1763,  1.3672, -2.7158], grad_fn=<ViewBackward0>)

In [ ]:
class Embedding(torch.nn.Module):
    def __init__(self,num_embeddings:int,embedding_dim:int,device=None,dtype=None):
        super().__init__()
        self.embedding_matrix=torch.nn.Parameter(torch.nn.init.trunc_normal_(torch.zeros(num_embeddings,embedding_dim),0,1))

    def forward(self,token_ids:torch.Tensor)->torch.Tensor:
        return self.embedding_matrix[token_ids]

In [63]:
# def root_mean_square(torch.Tensor)->torch.Tensor:
# x are the activations
from einops import reduce

d_model = 10
epsilon = 1e-5
x = torch.rand((4,12,d_model))
gain = torch.rand((4,12,d_model))
in_dtype = x.dtype
x2 = x**2
ssq = reduce(x2,'... last -> ... 1', 'sum')
rms = torch.sqrt(ssq/d_model + epsilon)
rms_norm = (x* gain) /rms
# out = rms_norm.to(in_dtype)
# out
print(ssq.shape)
print(rms.shape)
print(rms_norm.shape)


torch.Size([4, 12, 1])
torch.Size([4, 12, 1])
torch.Size([4, 12, 10])


In [35]:
num_embeddings=10
embedding_dim=5
embedding_matrix = torch.nn.init.trunc_normal_(torch.zeros(num_embeddings,embedding_dim),0,1)
print(f"Embedding matrix {embedding_matrix} of shape {embedding_matrix.shape}")
token_ids=torch.randint(low=0,high=3,size=(2,))
print(f"Token_ids {token_ids}")
test = embedding_matrix[token_ids]
print(f"Embeddings are {test}")

Embedding matrix tensor([[ 0.7618,  0.3471,  0.0363, -0.3564, -0.7420],
        [ 0.1797, -1.6826, -0.4914,  0.1416,  0.0442],
        [-0.0456,  0.4667,  0.0098, -1.9053,  0.8087],
        [-0.3326,  1.1250,  0.5572,  0.3468,  0.1888],
        [-0.5370, -1.7197, -1.5271, -1.4137,  0.8805],
        [ 1.3420, -0.3587, -0.6033, -0.4922,  1.6654],
        [ 0.7654, -0.9209, -1.2892,  0.9408,  1.2209],
        [ 0.0573, -0.2495,  1.9079, -0.5511,  1.1794],
        [ 0.1110,  0.0403, -0.1630, -0.9848, -0.8298],
        [-1.1684, -0.4662, -0.2611, -0.6399,  1.1354]]) of shape torch.Size([10, 5])
Token_ids tensor([1, 2])
Embeddings are tensor([[ 0.1797, -1.6826, -0.4914,  0.1416,  0.0442],
        [-0.0456,  0.4667,  0.0098, -1.9053,  0.8087]])


In [ ]:
from einops import rearrange,einsum
x = torch.randn(5)
W = torch.rand(4,5)
Y = x @ W.T

Y2 = einsum(x,W,"... d_in,d_out d_in -> ... d_out")
print(f"Original {Y} New {Y2}")

In [110]:
d_ff = 128 #Dimensionality of the up-project happening internally to your swiglu.
d_model=64 #Dimensionality of the feedforward input and output.
# d_ff = round(d_model/64) * 64
print(f"d_ff is {d_ff}")
# w1_weight (Float[Tensor, "d_ff d_model"]): Stored weights for W1
w1_weight = torch.rand((d_ff,d_model))
# w2_weight (Float[Tensor, "d_model d_ff"]): Stored weights for W2
w2_weight = torch.rand((d_model,d_ff))
# w3_weight (Float[Tensor, "d_ff d_model"]): Stored weights for W3
w3_weight = torch.rand((d_ff,d_model))
# in_features (Float[Tensor, "... d_model"]): Input embeddings to the feed-forward layer.
in_features = torch.rand((4,12,d_model))

print(f"In features is {in_features.shape}")
from cs336_basics.custom_modules import Linear
w1_x = Linear(d_model,d_ff)(in_features)
#w1_x = einsum(w1_weight,in_features,"d_ff d_model,... d_model->... d_ff ")
silu_w1_x = w1_x * torch.sigmoid(w1_x)
print(f"SiLU w1_x is {silu_w1_x.shape}")
w3_x = Linear(d_model,d_ff)(in_features)
# # w3_x = einsum(w3_weight,in_features,"d_ff d_model,... d_model ->... d_ff ")
# print(f"w3_x is {w3_x.shape}")
# silu_w1_x_w3_x = silu_w1_x * w3_x
# print(f"silu_w1_x_w3_x is {silu_w1_x_w3_x.shape}")
# # swiglu = einsum(w2_weight,silu_w1_x_w3_x," d_model d_ff, ... d_ff-> ... d_model " )
# swiglu = Linear(d_ff,d_model)(silu_w1_x_w3_x)
# print(f"swiglu is {swiglu.shape}")

d_ff is 128
In features is torch.Size([4, 12, 64])
SiLU w1_x is torch.Size([4, 12, 128])


In [ ]:
import torch
from einops import rearrange, repeat
batch_size = 4
# model size
embedding_dim=64
# a constant
theta=10000.0
max_seq_len=12
# the Q or the K matrix. Leading dims is batch-like
# last two dim must be sequence length and embedding dim
in_query_or_key = torch.rand((batch_size, max_seq_len, embedding_dim))

# q = torch.rand(batch_size,max_seq_len,embedding_dim)
logger.debug("in_query_or_key %s",in_query_or_key.shape)

def get_angles(embedding_dim:int,max_seq_len:int)->torch.Tensor:
    # create a tensor that stores for each pair of positions along the embedding dimension
    # their appropriate rotation angles where each element
    # is 1.0 / theta ^ (0/embeding_dim), 1.0 / theta ^ ((2)/embeding_dim),...
    angles = 1.0/10000 ** (torch.arange(0,embedding_dim,2,dtype=torch.float)/embedding_dim)
    logger.debug("angles %s of size %d ",angles[0:5],len(angles))
    # create a tensor of sequence positions [0..max_seq_len]
    token_positions = torch.arange(max_seq_len)
    # take the outer product so that each row of the output matrix is the angles vector scaled by each element of the position index. Lower coefficients (position index) means the angles are closer to zero.
    scaled_angles = torch.einsum("p,f->pf",token_positions,angles)
    logger.debug("Scaled inv freq %s",scaled_angles[:4,:4])
    # because we are working with pairs where each pair shares the same angle, duplicate these angles
    # and output a square matrix that has dimensions sequence length 
    return repeat(scaled_angles,"... f-> ... (f r)",r=2)

#print(fpe[:8,:8])
#print(f"q: {q[:2,:2]}")
def rotate_half(query_tokens:torch.Tensor)->torch.Tensor:
    logger.debug("Input tensor shape %s",query_tokens.shape)
    # take the last dimension and split it into chunks of 2
    query_tokens = rearrange(query_tokens,' ... (pos emb) -> ... pos emb', emb=2)
    logger.debug("Input tensor after splitting last dimension is shape %s",query_tokens.shape)
    # create two tensor slices, each slice containing matching element across the two chunks created above.
    # one slice contains odd elements the other even elements assuming 1-index. We lose a dimension.
    u1,u2 = query_tokens.unbind(dim=-1)
    logger.debug("Odd (1,3,5,...) slice is shape %s",u1.shape)
    logger.debug("Even (2,4,6,...) slice is shape %s",u2.shape)
    #print(f"u1: {u1[0]}\nu2: {u2[0]}")
    # Along the last dimension of interesting, stack the second slice to the first slice element wise, negating the even slice
    # We gain a dimension
    u3 = torch.stack((-u2,u1),dim=-1)
    logger.debug("Stacked slices is now shape %s",u3.shape)
    #print(f"u3 {u3[0]}")
    # collapse the last two dimensions into one
    merged_u3 = rearrange(u3,'... d r -> ... (d r)')
    logger.debug("Merged last two dimensions. Matrix is now dimension %s",merged_u3.shape)
    #print(f"merged_u3 {merged_u3}")
    return merged_u3

def do_rotate(query_tokens:torch.Tensor,angles:torch.Tensor)->torch.Tensor:
    # the second to the last dimension is the sequence length
    num_tokens = query_tokens.shape[-2]
    # we are concerned only with up to the sequence length in our datastructure
    pos_enc = angles[:num_tokens]
    #q*pos_enc.cos()
    rotation = query_tokens * pos_enc.cos()+(rotate_half(query_tokens) * pos_enc.sin())
    return rotation

angles = get_angles(embedding_dim,max_seq_len)
rotation = do_rotate(in_query_or_key,angles)
logger.debug("Rotation matrix is %s",rotation.shape)

In [5]:
from cs336_basics.custom_modules import RotaryPositionalEmbedding
import torch

batch_size = 4
# model size
embedding_dim=64
# a constant
theta=10000.0
max_seq_len=12


rope = RotaryPositionalEmbedding(theta=10000,d_k=embedding_dim,max_seq_len=max_seq_len,device=None)

# the Q or the K matrix. Leading dims is batch-like
# last two dim must be sequence length and embedding dim
in_query_or_key = torch.rand((batch_size, max_seq_len, embedding_dim))
token_positions = torch.arange(max_seq_len)
rope(in_query_or_key,token_positions)


cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,177 - angles tensor([1.0000, 0.7499, 0.5623, 0.4217, 0.3162]) of size 32 
cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,179 - Scaled inv freq tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.7499, 0.5623, 0.4217],
        [2.0000, 1.4998, 1.1247, 0.8434],
        [3.0000, 2.2497, 1.6870, 1.2651]])
cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,182 - Input tensor shape torch.Size([4, 12, 64])
cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,183 - Input tensor after splitting last dimension is shape torch.Size([4, 12, 32, 2])
cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,185 - Odd (1,3,5,...) slice is shape torch.Size([4, 12, 32])
cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,186 - Even (2,4,6,...) slice is shape torch.Size([4, 12, 32])
cs336_basics.custom_modules - DEBUG - 2026-10-08 12:34:57,187 - Stacked slices is now shape torch.Size([4, 12, 32, 2])
cs336_basi

In [14]:
from einops import repeat,rearrange
import numpy as np

# Start with a simple 2D matrix (height x width)
x = torch.tensor(np.array([[1, 2,3,4,  5,6,7,8]]))

print(f"x{x}")

output = rearrange(x,'... (d r) -> ... d r',r=2)
print(f"output {output} with shape {output.shape}")
u1,u2 =output.unbind(dim=-1)
print(f"u1 {u1}\nu2 {u2}")
u4 = torch.stack((-u2,u1),dim=-1)
print(f"u4 {u4}")
u5 = rearrange(u4,'... d r -> ... (d r)')
print(f"u5 {u5}")


xtensor([[1, 2, 3, 4, 5, 6, 7, 8]])
output tensor([[[1, 2],
         [3, 4],
         [5, 6],
         [7, 8]]]) with shape torch.Size([1, 4, 2])
u1 tensor([[1, 3, 5, 7]])
u2 tensor([[2, 4, 6, 8]])
u4 tensor([[[-2,  1],
         [-4,  3],
         [-6,  5],
         [-8,  7]]])
u5 tensor([[-2,  1, -4,  3, -6,  5, -8,  7]])


In [23]:
x = torch.Tensor(torch.rand((3,5)))
print(f"x is {x} with shape {x.shape}")
dim = -1
max = torch.max(x,dim,keepdim=True).values
print(f"max is {max} with shape {max.shape}")
x_smaller = torch.subtract(x,max)
print(f"x_smaller {x_smaller} with shape {x_smaller.shape}")
x_exp = torch.exp(x_smaller)
print(f"x_exp {x_exp} ")
denom = torch.sum(x_exp,dim = -1,keepdim=True)
print(f"denom {denom}")
softmax = x_exp/denom
print(f"softmax {softmax}")



x is tensor([[0.0463, 0.7230, 0.1843, 0.3211, 0.7613],
        [0.8705, 0.5905, 0.7423, 0.7720, 0.6262],
        [0.3111, 0.1426, 0.2613, 0.6959, 0.6889]]) with shape torch.Size([3, 5])
max is tensor([[0.7613],
        [0.8705],
        [0.6959]]) with shape torch.Size([3, 1])
x_smaller tensor([[-0.7150, -0.0383, -0.5770, -0.4401,  0.0000],
        [ 0.0000, -0.2800, -0.1282, -0.0985, -0.2443],
        [-0.3848, -0.5534, -0.4346,  0.0000, -0.0070]]) with shape torch.Size([3, 5])
x_exp tensor([[0.4892, 0.9625, 0.5616, 0.6439, 1.0000],
        [1.0000, 0.7558, 0.8797, 0.9062, 0.7833],
        [0.6806, 0.5750, 0.6475, 1.0000, 0.9930]]) 
denom tensor([[3.6572],
        [4.3249],
        [3.8961]])
softmax tensor([[0.1338, 0.2632, 0.1536, 0.1761, 0.2734],
        [0.2312, 0.1747, 0.2034, 0.2095, 0.1811],
        [0.1747, 0.1476, 0.1662, 0.2567, 0.2549]])


In [53]:
import math

batch_size = 4
seq_len = 12
embedding_dim = 64
# attention_len = 16
# number of queries
n = seq_len
# of keys/values set to seq_len because of self-attention
m = seq_len
# set to model size for now
d_k = embedding_dim
d_v = embedding_dim
q = torch.rand((batch_size,n,d_k))
k = torch.rand((batch_size,m,d_k))
v = torch.rand((batch_size,m,d_v))
# mask = torch.rand((batch_size,seq_len,attention_len))
mask = ~torch.randint(0,2,(batch_size,n,m)).bool()
# print(f"mask {mask}")
print(f"embedding dim {q.shape[-1]}")
from cs336_basics.custom_modules import Softmax
softmax = Softmax()
denom =  math.sqrt(embedding_dim)
print(f"denom {denom}")
qk = einsum(q,k,"... seq_len emb_dim , ... attn_len emb_dim -> ... seq_len attn_len ")
print(f"qk is {qk.shape} {qk}")
qk = qk/denom
print(f"mask is {mask}")
subtractor = lambda val: val - torch.inf
qk_masked = torch.where(~mask,subtractor(qk),qk)
print(f"qk2 is {qk_masked.shape} {qk_masked}")
softmaxed = softmax(qk_masked,dim=-1)
print(f"softmaxed = {softmaxed.shape}")
attn = einsum(softmaxed, v, "... seq_len attn_len ,... attn_len embedding_dim    ->... seq_len embedding_dim   ")
print(f"attn {attn}")

embedding dim 64
denom 8.0
qk is torch.Size([4, 12, 12]) tensor([[[13.7578,  9.7376, 13.0292, 13.9237, 13.2217, 12.5915, 12.3392,
          10.1687, 13.6445, 10.8006, 12.6502, 15.2084],
         [15.8337, 13.5173, 15.0729, 16.3572, 15.2027, 15.6025, 13.8136,
          13.2836, 16.2270, 12.1199, 15.4416, 16.3845],
         [17.0836, 16.1370, 15.4492, 17.3152, 15.9272, 16.7458, 15.6363,
          14.4985, 16.4302, 15.1065, 16.8128, 17.4728],
         [16.7700, 14.0040, 14.9129, 18.5462, 16.4801, 16.0993, 16.9966,
          15.5045, 17.2238, 14.4660, 15.7548, 18.1198],
         [18.8280, 16.3170, 16.2423, 17.8937, 18.2064, 19.3658, 16.2326,
          16.3869, 17.9112, 14.9559, 17.0971, 18.2869],
         [15.6735, 15.2994, 14.6236, 16.2277, 15.5096, 17.0582, 14.5189,
          14.2480, 15.1278, 12.8951, 16.1538, 16.0977],
         [16.2272, 15.4641, 15.4547, 16.1582, 16.2511, 16.6261, 14.9476,
          14.1072, 15.8317, 12.7081, 16.1965, 15.4452],
         [17.9727, 13.4773, 16.2600, 18.

In [42]:
# shapes are in_embeddings torch.Size([4, 12, 64]) d_model 64 num_heads 4 q torch.Size([64, 64]) k torch.Size([64, 64]) v torch.Size([64, 64]) o torch.Size([64, 64])

d_k = 8
d_v = 8
batch_size = 4
seq_len = 12
embedding_dim = 64
in_embeddings = torch.rand((batch_size,seq_len,embedding_dim))
heads = 4
q = torch.rand((heads*d_k,embedding_dim))
k = torch.rand((heads*d_k,embedding_dim))
v = torch.rand((heads*d_v,embedding_dim))
o = torch.rand((embedding_dim, heads*d_v))
mask = ~torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
token_positions = torch.arange(seq_len)

# take the last dimension and split it into chunks of 2
#q2 = rearrange(q,'(h d_k) embedding_dim -> h d_k embedding_dim ', h=heads)
# print(f"q {q.shape} q2 {q2.shape}")
# q_project should be heads,n,d_k
from cs336_basics.custom_modules import Linear
def get_sliced(weight:torch.Tensor,in_embeddings:torch.Tensor)->tuple[torch.Tensor]:
    project = einsum(weight,in_embeddings,"h_d_w embedding_dim,... seq_len embedding_dim -> ...  seq_len h_d_w")
    linear = Linear(heads*d_k, embedding_dim)
    linear.weight = torch.nn.Parameter(weight)
    project2 = linear(in_embeddings)
    # assert(project==project2)

    print(f"project {project.shape}")
    print(f"project2 {project2.shape}")
    project_sliced = rearrange(project,'... seq_len (h d_w) -> ... h seq_len d_w ', h=heads)
    print(f"project_sliced {project_sliced.shape}")
    return project_sliced
    # project_slices = project_sliced.unbind(dim=-3)
    # print(f"slices {len(project_slices)}")
    # for slice in project_slices:
    #      print(f"slice {slice.shape}")
    # return project_slices
# k_project = einsum(k,in_embeddings,"h embedding_dim,... seq_len embedding_dim -> ... seq_len  embedding_dim")
# v_project = einsum(v,in_embeddings,"h embedding_dim,... seq_len embedding_dim -> ... h seq_len  embedding_dim")
q_slices = get_sliced(q,in_embeddings)
from cs336_basics.custom_modules import RotaryPositionalEmbedding
rpe = RotaryPositionalEmbedding(10000,d_k,seq_len,None)
q_slices_rope = rpe(q_slices,token_positions)
print(f"q_slices_rope = {q_slices_rope.shape}")
k_slices = get_sliced(k,in_embeddings)
v_slices = get_sliced(v,in_embeddings)

from cs336_basics.custom_modules import ScaledDotProductAttention
sdpa = ScaledDotProductAttention()
multihead = sdpa(q_slices,k_slices,v_slices,mask)
print(f"multihead {multihead.shape}")
merged_multihead = rearrange(multihead,"... heads seq_len d_w -> ... seq_len (heads d_w)",heads=heads)
print(f"merged multihead {merged_multihead.shape}")
linear2 = Linear(d_v*heads,embedding_dim)
linear2.weight = torch.nn.Parameter(o)
o_project = linear2(merged_multihead)
o_project2 = einsum(o,merged_multihead,"emb_dim h_d_v , ... h_d_v -> ... emb_dim ")
print(f"o_project shape {o_project.shape}")
print(f"o_project2 shape {o_project2.shape}")
print(f"mask has nan? {torch.isnan(multihead).any()}")


cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,248 - angles tensor([1.0000, 0.1000, 0.0100, 0.0010]) of size 4 
cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,252 - Scaled inv freq tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03]])
cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,255 - Input tensor shape torch.Size([4, 4, 12, 8])
cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,258 - Input tensor after splitting last dimension is shape torch.Size([4, 4, 12, 4, 2])
cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,260 - Odd (1,3,5,...) slice is shape torch.Size([4, 4, 12, 4])
cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,260 - Even (2,4,6,...) slice is shape torch.Size([4, 4, 12, 4])
cs336_basics.custom_modules - DEBUG - 2026-10-09 20:10:14,261 - St

project torch.Size([4, 12, 32])
project2 torch.Size([4, 12, 32])
project_sliced torch.Size([4, 4, 12, 8])
q_slices_rope = torch.Size([4, 4, 12, 8])
project torch.Size([4, 12, 32])
project2 torch.Size([4, 12, 32])
project_sliced torch.Size([4, 4, 12, 8])
project torch.Size([4, 12, 32])
project2 torch.Size([4, 12, 32])
project_sliced torch.Size([4, 4, 12, 8])
multihead torch.Size([4, 4, 12, 8])
merged multihead torch.Size([4, 12, 32])
o_project shape torch.Size([4, 12, 64])
o_project2 shape torch.Size([4, 12, 64])
mask has nan? False


In [47]:

# d_model 64 num_heads 4 d_ff 128 max_seq_len 16 theta 10000.0
# in features shape torch.Size([4, 12, 64])
# weight attn.q_proj.weight has shape torch.Size([64, 64])
# weight attn.k_proj.weight has shape torch.Size([64, 64])
# weight attn.v_proj.weight has shape torch.Size([64, 64])
# weight attn.output_proj.weight has shape torch.Size([64, 64])
# weight ffn.w1.weight has shape torch.Size([128, 64])
# weight ffn.w2.weight has shape torch.Size([64, 128])
# weight ffn.w3.weight has shape torch.Size([128, 64])
# weight ln1.weight has shape torch.Size([64])
# weight ln2.weight has shape torch.Size([64])
pos_ids = torch.arange(end=12).shape
pos_ids = rearrange(pos_ids, "seq -> 1 seq")
pos_ids


RuntimeError: Tensor type unknown to einops <class 'torch.Size'>